# Assignment 4 — Cognitive FAQ System Using Pandas

**Jiya Mahajan | Roll No. 1024170296 | Batch 3Q31 / 3Q3A**


## Q1


In [ ]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]
personalized_entries = [
    {"question": "how can I check the status of a recent payment", "answer": "Open Billing > Payment History to view the latest status.", "keywords": "payment status history transaction", "category": "billing"},
    {"question": "how can I request a copy of my payment receipt", "answer": "Open Billing > Payment History and select Download Receipt.", "keywords": "receipt payment billing", "category": "billing"},
]
df = pd.DataFrame(fixed_entries + personalized_entries)
df

## Q2


In [ ]:
import re
def score_query(query, knowledge_base=df):
    terms = set(re.findall(r"[a-z0-9]+", query.lower()))
    scored = []
    for _, row in knowledge_base.iterrows():
        searchable = set(re.findall(r"[a-z0-9]+", row["question"] + " " + row["keywords"].replace(",", " ").lower()))
        score = len(terms & searchable) / len(terms) if terms else 0.0
        if score > 0:
            scored.append((score, row))
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored

for score, row in score_query("fee payment"):
    print(f"{score:.2f}", row["question"], "->", row["answer"])

## Q3


In [ ]:
def same_category(category_name, data):
    return data.loc[data["category"].str.lower() == category_name.lower(), "question"]

personalized_category = df.iloc[4]["category"]
print("Personalized category:", personalized_category)
print(same_category(personalized_category, df).to_string(index=False))

## Q4


In [ ]:
entry_index = 4
new_keyword = input("Enter a new keyword to add to the personalized FAQ: ").strip()
if new_keyword:
    existing = df.at[entry_index, "keywords"].split()
    if new_keyword.lower() not in {word.lower() for word in existing}:
        df.at[entry_index, "keywords"] = df.at[entry_index, "keywords"] + " " + new_keyword
df.to_csv("1024170296_faq_data.csv", index=False)
df

## Q5


In [ ]:
df.groupby("category").size().rename("entry_count").to_frame()

## Q6


In [ ]:
def best_matches(query, knowledge_base=df):
    scored = score_query(query, knowledge_base)
    if not scored:
        print("No FAQ matched the query.")
        return pd.DataFrame()
    top_score = scored[0][0]
    ties = [row for score, row in scored if score == top_score]
    result = pd.DataFrame(ties).reset_index(drop=True)
    print(f"Highest confidence: {top_score:.2f}; matching entries: {len(result)}")
    return result

print("Tie demonstration:")
best_matches("fee")
print("\nNon-tie demonstration:")
best_matches("password reset login")